# Chart Summaries

Last updated by Michael Harries and Claude Code, Mar 20, 2026.

Part of the IEEE RAS robotics industry report generation pipeline.

---

Generates a single `chart_summaries.txt` file containing structured statistics
for every chart produced by the main Robotics_Industry_Analysis notebook and
the Missing_Charts_Analysis notebook. Bullet-pointed stats serve as raw material
for publication prose.

## How It Works

1. Upload the same two Tracxn Excel files used by the main notebook
2. The data pipeline (cells 2–4) is identical to the main notebook
3. Cell 5 defines per-chart-type summary functions that compute key statistics
4. Cell 6 iterates through all chart names and writes a single output file

## Output

Single file: `no_title_charts/chart_summaries.txt`

The file has two sections:

1. **HIGHLIGHTS** — Striking findings auto-detected by threshold rules:
   - Concentration: HHI ≥ 2500, top share ≥ 60%
   - Growth: |CAGR| ≥ 30%, |year-over-year change| ≥ 100%
   - Deadpool rate ≥ 25%, funded rate ≤ 30%
   - Mean/median round-size ratio ≥ 5x

2. **FULL CHART SUMMARIES** — All stats per chart, with highlighted lines
   marked with `**` prefix

```
======================================================================
HIGHLIGHTS — Striking Findings
======================================================================

[robotics_funding_by_year_2000_onwards]
** CAGR (2000-2025): +42.3%
** Mean/median round size ratio: 8.2x (heavy right-tail skew)
...

======================================================================
FULL CHART SUMMARIES
======================================================================

--------------------------------------------------
Chart: stacked_funding
Scope: Global
Period: 2000-2025

- Total disclosed funding: $X.XB across N rounds
** CAGR (2000-2025): +42.3%
- ...
```

## Required Inputs

Same as the main notebook:

| File | Key columns |
|---|---|
| Companies export | `Subcategory`, `Domain Name`, `Category`, `Country`, `Founded Year`, etc. |
| Funding rounds export | `Domain Name`, `Round Amount (in USD)` or `Round Amount (USD)`, `Round Date` |


In [ ]:
# Cell 2: Setup & Imports
!pip install -q openpyxl

%config InlineBackend.figure_format = 'retina'

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter
import pandas as pd
import numpy as np
import os
import zipfile
import gc
from google.colab import files

In [ ]:
# Cell 3: Configuration

# --- Workflow Control ---
# Set to True to skip Tracxn file uploads and only generate the global VC chart
SKIP_TRACXN_UPLOAD = False
# Set to True to skip the global VC CSV upload and only generate Tracxn charts
SKIP_VC_UPLOAD = False

# Year ranges
START_YEAR_ALL = 1900
START_YEAR_DEFAULT = 2000

# Funding thresholds
ROUND_SIZE_THRESHOLD = 100_000_000  # $100M boundary for stacked funding charts

# Display limits
TOP_SECTORS_COUNT = 7  # Sectors shown individually before 'Other' bucket

# Output
CHART_OUTPUT_DIR = 'no_title_charts'

# --- Color Palette (Paul Tol Colorblind-Friendly) ---
PAUL_TOL_PRIMARY = ['#4477AA', '#EE6677', '#228833', '#CCBB44', '#66CCEE', '#AA3377', '#BBBBBB']

PAUL_TOL_EXTENDED = PAUL_TOL_PRIMARY + [
    '#332288', '#88CCEE', '#44AA99', '#117733',
    '#999933', '#DDCC77', '#CC6677', '#882255', '#AA4499'
]

BINARY_STATUS_COLORS = ['#88CCEE', '#332288']  # Paul Tol: light blue (No), dark blue (Yes)

# --- Country Standardization ---
COUNTRY_STANDARDIZATION = {
    'USA': 'United States',
    'Republic of Korea': 'South Korea',
    'United Arab Emirates': 'UAE',
}

# --- Second-Tier Region Mapping ---
# All country names use POST-standardization values
SECOND_TIER_REGIONS = {
    'USA': ['United States'],
    'Canada': ['Canada'],
    'China': ['China'],
    'United Kingdom': ['United Kingdom'],
    'India': ['India'],
    'APAC': [
        'Japan', 'Kazakhstan', 'Nepal', 'Pakistan', 'Australia',
        'Bangladesh', 'New Zealand', 'South Korea', 'Singapore',
        'Indonesia', 'Malaysia', 'Thailand', 'Vietnam',
        'Philippines', 'Sri Lanka', 'Taiwan'
    ],
    'Middle East': [
        'Jordan', 'Kuwait', 'Oman', 'Armenia', 'Israel',
        'Turkey', 'UAE', 'Iran', 'Lebanon', 'Saudi Arabia'
    ],
    'Africa': [
        'Senegal', 'Rwanda', 'Ghana', 'Tunisia', 'Morocco',
        'South Africa', 'Egypt', 'Nigeria', 'Kenya'
    ],
    'Western Europe': [
        'Germany', 'France', 'Spain', 'Italy', 'Netherlands',
        'Austria', 'Belgium', 'Switzerland', 'Luxembourg',
        'Portugal', 'Ireland'
    ],
    'Eastern Europe': [
        'Belarus', 'Serbia', 'Bulgaria', 'Ukraine', 'Poland',
        'Russia', 'Czech Republic', 'Slovakia', 'Slovenia',
        'Croatia', 'Cyprus', 'Greece', 'Hungary', 'Romania'
    ],
    'Nordics and Baltics': [
        'Sweden', 'Norway', 'Finland', 'Denmark', 'Iceland',
        'Estonia', 'Latvia', 'Lithuania'
    ],
    'Americas (No Canada and USA)': [
        'Guatemala', 'Puerto Rico', 'Costa Rica', 'Uruguay',
        'Panama', 'Ecuador', 'Mexico', 'Brazil', 'Argentina',
        'Chile', 'Colombia', 'Peru'
    ],
}

# --- Reverse Region Lookup ---
COUNTRY_TO_REGION = {}
for region, countries in SECOND_TIER_REGIONS.items():
    for country in countries:
        COUNTRY_TO_REGION[country] = region

# --- Investment Type Analysis ---
INVESTMENT_TYPE_START_YEAR = 2011
TOP_INVESTMENT_TYPES = 12  # Remainder grouped as "Other"

# --- Missing Funding Analysis ---
MISSING_FUNDING_START_YEAR = 2011
MIN_EVENTS_THRESHOLD = 20
TOP_COUNTRIES_COUNT = 15  # For missing funding by-country chart

# Macro-region grouping (for missing funding region chart)
# Maps macro-region display name -> list of SECOND_TIER_REGIONS keys
MACRO_REGION_GROUPS = {
    'Americas': ['USA', 'Canada', 'Americas (No Canada and USA)'],
    'Europe/Africa': ['United Kingdom', 'Western Europe', 'Eastern Europe',
                      'Nordics and Baltics', 'Middle East', 'Africa'],
    'Asia/Pacific': ['China', 'India', 'APAC'],
}

# Build country -> macro-region lookup from SECOND_TIER_REGIONS
COUNTRY_TO_MACRO_REGION = {}
for macro, second_tiers in MACRO_REGION_GROUPS.items():
    for st in second_tiers:
        for country in SECOND_TIER_REGIONS[st]:
            COUNTRY_TO_MACRO_REGION[country] = macro

In [ ]:
# Cell 4: Data Upload & Cleaning
import re
from datetime import datetime

# --- Cleaning Functions ---

def clean_amount(amount):
    """Clean and convert amount to float. Returns 0.0 on failure."""
    if pd.isna(amount) or amount is None:
        return 0.0
    if isinstance(amount, (int, float)):
        return float(amount)
    if isinstance(amount, str):
        amount = amount.replace(',', '').replace('$', '').replace(' ', '')
        if amount == '':
            return 0.0
        try:
            return float(amount)
        except ValueError:
            return 0.0
    return 0.0

def extract_year(date_str):
    """Extract year from various date formats. Valid range: 1800-2030."""
    if pd.isna(date_str) or date_str is None:
        return None
    date_str = str(date_str).strip()
    if date_str == '' or date_str.lower() == 'none':
        return None

    # Handle bare year (4 digits)
    if len(date_str) == 4 and date_str.isdigit():
        year = int(date_str)
        return year if 1800 <= year <= 2030 else None

    # Handle "Jan 01, 2020" format (Tracxn)
    if ',' in date_str:
        try:
            year = datetime.strptime(date_str.strip(), '%b %d, %Y').year
            return year if 1800 <= year <= 2030 else None
        except (ValueError, TypeError):
            pass

    # Try standard date formats
    for fmt in ['%Y-%m-%d', '%m/%d/%Y', '%d/%m/%Y', '%b %Y', '%B %Y']:
        try:
            year = datetime.strptime(date_str, fmt).year
            return year if 1800 <= year <= 2030 else None
        except ValueError:
            continue

    # Regex fallback
    match = re.search(r'\b(18|19|20)\d{2}\b', date_str)
    if match:
        year = int(match.group())
        return year if 1800 <= year <= 2030 else None

    return None

def clean_category(category):
    """Clean category names - take first if comma-separated."""
    if pd.isna(category) or category is None:
        return 'Unknown'
    category = str(category).strip()
    if category == '':
        return 'Unknown'
    if ',' in category:
        category = category.split(',')[0].strip()
    return category

def clean_country(country):
    """Clean country name - take first if comma-separated, apply standardization."""
    if pd.isna(country) or country is None:
        return 'Unknown'
    country = str(country).strip()
    if country == '':
        return 'Unknown'
    if ',' in country:
        country = country.split(',')[0].strip()
    return COUNTRY_STANDARDIZATION.get(country, country)

if SKIP_TRACXN_UPLOAD:
    print("Skipping Tracxn file upload (SKIP_TRACXN_UPLOAD = True).")
    print("Only the global VC totals chart will be available.")
else:
    # --- File Upload ---
    print("Upload two Excel files: one companies export and one funding rounds export from Tracxn.")
    uploaded = files.upload()
    filenames = list(uploaded.keys())
    assert len(filenames) == 2, f"Expected 2 files, got {len(filenames)}: {filenames}"

    dfs = {}
    for fn in filenames:
        dfs[fn] = pd.read_excel(fn, engine='openpyxl')
        print(f"  Loaded {fn}: {dfs[fn].shape[0]} rows, {dfs[fn].shape[1]} columns")

    # --- Auto-Detection ---
    companies_file = None
    funding_file = None

    for fn, df in dfs.items():
        if 'Subcategory' in df.columns:
            if companies_file is not None:
                raise ValueError("Both files contain 'Subcategory'. Please upload one companies file and one funding rounds file.")
            companies_file = fn
        else:
            funding_file = fn

    if companies_file is None:
        all_cols = {fn: list(df.columns) for fn, df in dfs.items()}
        raise ValueError(f"Neither uploaded file contains a 'Subcategory' column. "
                         f"The companies file from Tracxn must include this column. "
                         f"Available columns: {all_cols}")

    if funding_file is None:
        funding_file = [fn for fn in filenames if fn != companies_file][0]

    print(f"\nDetected companies file: {companies_file}")
    print(f"Detected funding file:  {funding_file}")

    companies_raw = dfs[companies_file]
    funding_raw = dfs[funding_file]

    # --- Validation ---
    assert 'Domain Name' in companies_raw.columns, \
        f"Companies file missing 'Domain Name'. Columns: {list(companies_raw.columns)}"
    assert 'Domain Name' in funding_raw.columns, \
        f"Funding file missing 'Domain Name'. Columns: {list(funding_raw.columns)}"

    # Round amount column may be named differently across Tracxn export versions
    ROUND_AMOUNT_COL = None
    for candidate in ['Round Amount (in USD)', 'Round Amount (USD)']:
        if candidate in funding_raw.columns:
            ROUND_AMOUNT_COL = candidate
            break
    assert ROUND_AMOUNT_COL is not None, \
        f"Funding file missing round amount column. Looked for 'Round Amount (in USD)' or 'Round Amount (USD)'. Columns: {list(funding_raw.columns)}"
    print(f"Using round amount column: '{ROUND_AMOUNT_COL}'")

    assert 'Round Date' in funding_raw.columns, \
        f"Funding file missing 'Round Date'. Columns: {list(funding_raw.columns)}"

    # --- Inner Join ---
    merged_df = pd.merge(companies_raw, funding_raw, on='Domain Name', how='inner',
                         suffixes=('_Companies', '_Funding'))
    print(f"\nMerged: {merged_df.shape[0]} rows (inner join on Domain Name)")

    # --- Apply Cleaning ---
    # Determine column names (may have suffixes from merge)
    cat_col = 'Category_Companies' if 'Category_Companies' in merged_df.columns else 'Category'
    country_col = 'Country_Companies' if 'Country_Companies' in merged_df.columns else 'Country'

    merged_df['amount_usd'] = merged_df[ROUND_AMOUNT_COL].apply(clean_amount)
    merged_df['year'] = merged_df['Round Date'].apply(extract_year)
    merged_df['category_clean'] = merged_df[cat_col].apply(clean_category)
    merged_df['country_clean'] = merged_df[country_col].apply(clean_country)
    merged_df['investment_type'] = merged_df['Round Name'].fillna('Unspecified').replace('', 'Unspecified').str.strip()

    # Note: Rows with category_clean='Unknown' or country_clean='Unknown' are kept.
    # They contribute to global totals and charts that don't filter by that dimension.
    # Scope filters in chart functions naturally exclude them from dimension-specific
    # charts (e.g., a region filter won't match 'Unknown' country).

    # --- Filter to analysis_df (disclosed amounts, valid years) ---
    analysis_df = merged_df[(merged_df['amount_usd'] > 0) & (merged_df['year'].notna())].copy()
    analysis_df['year'] = analysis_df['year'].astype(int)

    # --- all_events_df (all events with valid years, including undisclosed) ---
    all_events_df = merged_df[merged_df['year'].notna()].copy()
    all_events_df['year'] = all_events_df['year'].astype(int)

    # --- companies_df (companies-only, for founded-year charts) ---
    companies_df = companies_raw.copy()
    companies_df['founded_year'] = companies_df['Founded Year'].apply(extract_year)
    companies_df = companies_df[companies_df['founded_year'].notna()].copy()
    companies_df['founded_year'] = companies_df['founded_year'].astype(int)
    companies_df = companies_df[(companies_df['founded_year'] >= 1800) & (companies_df['founded_year'] <= 2030)]
    companies_df['category_clean'] = companies_df['Category'].apply(clean_category)
    companies_df['country_clean'] = companies_df['Country'].apply(clean_country)

    # --- Summary Statistics ---
    print(f"\n{'='*50}")
    print(f"DATA PIPELINE SUMMARY")
    print(f"{'='*50}")
    print(f"Companies uploaded:    {companies_raw.shape[0]:,}")
    print(f"Funding rounds uploaded: {funding_raw.shape[0]:,}")
    print(f"Merged rows:           {merged_df.shape[0]:,}")
    print(f"analysis_df rows:      {analysis_df.shape[0]:,} (disclosed amounts, valid years)")
    print(f"all_events_df rows:    {all_events_df.shape[0]:,} (all events, valid years)")
    print(f"companies_df rows:     {companies_df.shape[0]:,} (valid founded year)")
    print(f"Year range:            {int(analysis_df['year'].min())} - {int(analysis_df['year'].max())}")
    print(f"Unique companies:      {merged_df['Domain Name'].nunique():,}")
    print(f"Unique sectors:        {analysis_df['category_clean'].nunique()}")
    print(f"{'='*50}")

In [ ]:
# Cell 5: Summary Helper Functions

import re as _re


# Accumulator for all summaries (written to single file at end)
all_summaries = []    # list of (chart_name, lines, highlights)


def detect_highlights(lines):
    """Scan bullet lines for striking findings. Returns list of highlight strings.

    Thresholds:
    - Concentration: top-1 share > 50%, HHI >= 2500
    - Growth: |CAGR| > 30%, |year-over-year change| > 100%
    - Dominance: single category/type/region > 60% share
    - Decline: deadpool rate > 25%, funded rate < 30%
    - Skew: mean/median ratio > 5x (heavy right-tail)
    """
    highlights = []
    for line in lines:
        if not line.startswith('- '):
            continue

        # --- Concentration ---
        if 'HHI:' in line:
            import re as _hl_re
            m = _hl_re.search(r'HHI:\s*([\d.]+)', line)
            if m and float(m.group(1)) >= 2500:
                highlights.append(line.replace('- ', '** '))

        # Top-1 or top-3 share > 60%
        if ('of total' in line or 'concentration' in line.lower()):
            import re as _hl_re
            pcts = _hl_re.findall(r'([\d.]+)%', line)
            for p in pcts:
                if float(p) >= 60:
                    highlights.append(line.replace('- ', '** '))
                    break

        # --- Growth / Decline ---
        if 'CAGR' in line:
            import re as _hl_re
            m = _hl_re.search(r'([+-]?[\d.]+)%', line)
            if m and abs(float(m.group(1))) >= 30:
                highlights.append(line.replace('- ', '** '))

        if '% change' in line:
            import re as _hl_re
            m = _hl_re.search(r'([+-]?[\d.]+)%\s*change', line)
            if m and abs(float(m.group(1))) >= 100:
                highlights.append(line.replace('- ', '** '))

        # --- Deadpool / Funded rates ---
        if 'deadpool' in line.lower() and '%' in line:
            import re as _hl_re
            pcts = _hl_re.findall(r'([\d.]+)%', line)
            for p in pcts:
                if float(p) >= 25:
                    highlights.append(line.replace('- ', '** '))
                    break

        if 'funded' in line.lower() and 'Not funded' not in line:
            if 'Scope' not in line and 'Chart' not in line:
                import re as _hl_re
                pcts = _hl_re.findall(r'([\d.]+)%', line)
                for p in pcts:
                    if float(p) <= 30 and 'funded' in line.lower():
                        highlights.append(line.replace('- ', '** '))
                        break

        # --- Skew: mean >> median ---
        if 'Mean' in line and 'Median' not in line:
            # Will be caught by companion median line below
            pass

    # Cross-line check: mean vs median skew
    mean_val = None
    median_val = None
    for line in lines:
        if 'Mean' in line and 'round size' in line.lower():
            import re as _hl_re
            m = _hl_re.search(r'\$([\d.]+)([BMK])', line)
            if m:
                v = float(m.group(1))
                mult = {'B': 1e9, 'M': 1e6, 'K': 1e3}.get(m.group(2), 1)
                mean_val = v * mult
        if 'Median' in line and 'round size' in line.lower():
            import re as _hl_re
            m = _hl_re.search(r'\$([\d.]+)([BMK])', line)
            if m:
                v = float(m.group(1))
                mult = {'B': 1e9, 'M': 1e6, 'K': 1e3}.get(m.group(2), 1)
                median_val = v * mult
    if mean_val and median_val and median_val > 0:
        ratio = mean_val / median_val
        if ratio >= 5:
            highlights.append(f'** Mean/median round size ratio: {ratio:.1f}x (heavy right-tail skew)')

    return highlights


def save_summary(filename, lines):
    """Accumulate summary lines for single-file output."""
    highlights = detect_highlights(lines)
    all_summaries.append((filename, lines, highlights))
    n_hl = len(highlights)
    if n_hl > 0:
        print(f'  {filename}: {n_hl} highlight(s)')


def fmt_usd(amount):
    """Format a USD amount as human-readable string."""
    if amount >= 1e9:
        return f'${amount/1e9:.1f}B'
    elif amount >= 1e6:
        return f'${amount/1e6:.0f}M'
    elif amount >= 1e3:
        return f'${amount/1e3:.0f}K'
    else:
        return f'${amount:,.0f}'


def fmt_pct(value):
    """Format a percentage value."""
    return f'{value:.1f}%'


def safe_cagr(start_val, end_val, years):
    """Compute CAGR safely, returning None if not meaningful."""
    if years <= 0 or start_val <= 0 or end_val <= 0:
        return None
    return ((end_val / start_val) ** (1 / years) - 1) * 100


def hhi(shares):
    """Compute Herfindahl-Hirschman Index from percentage shares."""
    return sum(s**2 for s in shares)


def hhi_label(hhi_val):
    """Classify HHI concentration level."""
    if hhi_val >= 2500:
        return 'highly concentrated'
    elif hhi_val >= 1500:
        return 'moderately concentrated'
    else:
        return 'competitive'


def sanitize_filename(name):
    """Convert display name to filename-safe string."""
    name = name.replace('&', 'and')
    name = name.replace('/', '_')
    name = name.replace(',', '')
    name = name.replace('(', '').replace(')', '')
    name = name.replace(' ', '_')
    name = _re.sub(r'[^a-zA-Z0-9_.]', '', name)
    return name


def is_yes(series):
    """Check if a pandas Series contains 'Yes' values (case-insensitive)."""
    return series.astype(str).str.strip().str.lower() == 'yes'


# =============================================================================
# Per-chart-type summary functions
# =============================================================================

def summarize_companies_founded(companies_df, scope_filter, scope_name, start_year=2000):
    """Summary for companies-founded line/bar/outcomes/sector/funding-status charts."""
    df = scope_filter(companies_df) if scope_filter else companies_df
    if len(df) == 0:
        return None

    total = len(df)
    recent = df[df['founded_year'] >= start_year]
    yearly = df.groupby('founded_year').size()
    peak_year = int(yearly.idxmax())
    peak_count = int(yearly.max())

    lines = [
        f'Chart: companies_founded',
        f'Scope: {scope_name}',
        f'Period: {int(df["founded_year"].min())}-{int(df["founded_year"].max())}',
        '',
        f'- Total companies: {total:,}',
        f'- Companies founded {start_year}+: {len(recent):,}',
        f'- Peak founding year: {peak_year} ({peak_count:,} companies)',
    ]

    # Top 3 sectors
    top_sectors = df['category_clean'].value_counts().head(3)
    for i, (sector, count) in enumerate(top_sectors.items(), 1):
        pct = count / total * 100
        lines.append(f'- #{i} sector: {sector} ({count:,} companies, {fmt_pct(pct)})')

    # Funded %
    if 'Is Funded' in df.columns:
        funded = is_yes(df['Is Funded']).sum()
        lines.append(f'- Funded: {funded:,} ({fmt_pct(funded/total*100)})')

    # Deadpooled %
    if 'Is Deadpooled' in df.columns:
        dead = is_yes(df['Is Deadpooled']).sum()
        lines.append(f'- Deadpooled: {dead:,} ({fmt_pct(dead/total*100)})')

    return lines


def summarize_stacked_funding(analysis_df, all_events_df, scope_filter, scope_name, start_year):
    """Summary for stacked funding (round size) charts."""
    adf = scope_filter(analysis_df) if scope_filter else analysis_df
    edf = scope_filter(all_events_df) if scope_filter else all_events_df
    adf = adf[adf['year'] >= start_year]
    edf = edf[edf['year'] >= start_year]
    if len(adf) == 0:
        return None

    total_funding = adf['amount_usd'].sum()
    total_rounds_disclosed = len(adf)
    total_rounds_all = len(edf)
    max_year = int(adf['year'].max())
    min_year = int(adf['year'].min())

    # Round size split
    large = adf[adf['amount_usd'] >= ROUND_SIZE_THRESHOLD]['amount_usd'].sum()
    small = total_funding - large
    pct_large = large / total_funding * 100 if total_funding > 0 else 0

    # Peak year
    yearly = adf.groupby('year')['amount_usd'].sum()
    peak_year = int(yearly.idxmax())
    peak_amount = yearly.max()

    lines = [
        f'Chart: stacked_funding',
        f'Scope: {scope_name}',
        f'Period: {start_year}-{max_year}',
        '',
        f'- Total disclosed funding: {fmt_usd(total_funding)} across {total_rounds_disclosed:,} disclosed rounds',
        f'- Total funding events (incl. undisclosed): {total_rounds_all:,}',
        f'- Rounds >= $100M: {fmt_usd(large)} ({fmt_pct(pct_large)} of total)',
        f'- Rounds < $100M: {fmt_usd(small)} ({fmt_pct(100-pct_large)} of total)',
        f'- Peak year: {peak_year} ({fmt_usd(peak_amount)})',
    ]

    # CAGR if enough years
    num_years = max_year - min_year
    if num_years >= 3:
        cagr = safe_cagr(yearly.get(min_year, 0), yearly.get(max_year, 0), num_years)
        if cagr is not None:
            lines.append(f'- CAGR ({min_year}-{max_year}): {cagr:+.1f}%')

    # Mean/median round size
    mean_round = adf['amount_usd'].mean()
    median_round = adf['amount_usd'].median()
    lines.append(f'- Mean disclosed round size: {fmt_usd(mean_round)}')
    lines.append(f'- Median disclosed round size: {fmt_usd(median_round)}')

    return lines


def summarize_total_funding(analysis_df, scope_name, start_year):
    """Summary for total_global_funding_by_year chart (dual-axis bars+line)."""
    df = analysis_df[analysis_df['year'] >= start_year]
    if len(df) == 0:
        return None

    total = df['amount_usd'].sum()
    total_rounds = len(df)
    max_year = int(df['year'].max())

    yearly = df.groupby('year')['amount_usd'].sum()
    peak_year = int(yearly.idxmax())
    peak_val = yearly.max()

    yearly_counts = df.groupby('year').size()
    peak_count_year = int(yearly_counts.idxmax())
    peak_count = int(yearly_counts.max())

    mean_round = df['amount_usd'].mean()
    median_round = df['amount_usd'].median()

    lines = [
        f'Chart: total_global_funding_by_year',
        f'Scope: {scope_name}',
        f'Period: {start_year}-{max_year}',
        '',
        f'- Total disclosed funding: {fmt_usd(total)}',
        f'- Total disclosed rounds: {total_rounds:,}',
        f'- Peak funding year: {peak_year} ({fmt_usd(peak_val)})',
        f'- Peak round-count year: {peak_count_year} ({peak_count:,} rounds)',
        f'- Mean round size: {fmt_usd(mean_round)}',
        f'- Median round size: {fmt_usd(median_round)}',
    ]

    return lines


def summarize_funding_yearly_bar(df, metric, scope_filter, scope_name, start_year):
    """Summary for simple funding amount or event count bar charts (Missing Charts notebook)."""
    filtered = scope_filter(df) if scope_filter else df
    filtered = filtered[filtered['year'] >= start_year]
    if len(filtered) == 0:
        return None

    max_year = int(filtered['year'].max())

    if metric == 'amount':
        yearly = filtered.groupby('year')['amount_usd'].sum()
        total = filtered['amount_usd'].sum()
        peak_year = int(yearly.idxmax())
        peak_val = yearly.max()
        total_rounds = len(filtered)

        lines = [
            f'Chart: funding_yearly_bar (amount)',
            f'Scope: {scope_name}',
            f'Period: {start_year}-{max_year}',
            '',
            f'- Total disclosed funding: {fmt_usd(total)} across {total_rounds:,} rounds',
            f'- Peak year: {peak_year} ({fmt_usd(peak_val)})',
        ]

        # Recent trend (last 3 years)
        recent_years = sorted(yearly.index)[-3:]
        if len(recent_years) >= 2:
            recent_vals = [yearly[y] for y in recent_years]
            trend = 'increasing' if recent_vals[-1] > recent_vals[-2] else 'decreasing'
            lines.append(f'- Recent trend: {trend} ({", ".join(str(y) for y in recent_years)})')
            for y in recent_years:
                lines.append(f'  - {y}: {fmt_usd(yearly[y])}')

        # CAGR
        num_years = max_year - start_year
        if num_years >= 3:
            cagr = safe_cagr(yearly.get(start_year, 0), yearly.get(max_year, 0), num_years)
            if cagr is not None:
                lines.append(f'- CAGR ({start_year}-{max_year}): {cagr:+.1f}%')

    else:  # count
        yearly = filtered.groupby('year').size()
        total = len(filtered)
        peak_year = int(yearly.idxmax())
        peak_count = int(yearly.max())

        lines = [
            f'Chart: funding_yearly_bar (count)',
            f'Scope: {scope_name}',
            f'Period: {start_year}-{max_year}',
            '',
            f'- Total funding events: {total:,}',
            f'- Peak year: {peak_year} ({peak_count:,} events)',
        ]

        recent_years = sorted(yearly.index)[-3:]
        if len(recent_years) >= 2:
            recent_vals = [yearly[y] for y in recent_years]
            trend = 'increasing' if recent_vals[-1] > recent_vals[-2] else 'decreasing'
            lines.append(f'- Recent trend: {trend} ({", ".join(str(y) for y in recent_years)})')
            for y in recent_years:
                lines.append(f'  - {y}: {int(yearly[y]):,} events')

    return lines


def summarize_missing_funding(all_events_df, start_year, group_col, scope_name):
    """Summary for missing funding analysis charts."""
    df = all_events_df[all_events_df['year'] >= start_year]
    if len(df) == 0:
        return None

    total_events = len(df)
    missing = (df['amount_usd'] == 0).sum() if 'amount_usd' in df.columns else 0
    overall_pct = missing / total_events * 100

    max_year = int(df['year'].max())

    lines = [
        f'Chart: missing_funding',
        f'Scope: {scope_name} (by {group_col})',
        f'Period: {start_year}-{max_year}',
        '',
        f'- Total funding events: {total_events:,}',
        f'- Missing (undisclosed) amount: {missing:,} ({fmt_pct(overall_pct)})',
    ]

    if group_col == 'year':
        yearly_total = df.groupby('year').size()
        yearly_missing = df[df['amount_usd'] == 0].groupby('year').size()
        yearly_pct = (yearly_missing / yearly_total * 100).dropna()
        if len(yearly_pct) > 0:
            worst_year = int(yearly_pct.idxmax())
            best_year = int(yearly_pct.idxmin())
            lines.append(f'- Worst year: {worst_year} ({fmt_pct(yearly_pct[worst_year])} missing)')
            lines.append(f'- Best year: {best_year} ({fmt_pct(yearly_pct[best_year])} missing)')
            # Trend
            recent = sorted(yearly_pct.index)[-3:]
            if len(recent) >= 2:
                vals = [yearly_pct[y] for y in recent]
                direction = 'improving' if vals[-1] < vals[-2] else 'worsening'
                lines.append(f'- Recent trend: {direction}')
    elif group_col in ('country_clean', 'category_clean', 'investment_type', 'macro_region'):
        if group_col == 'macro_region':
            df = df.copy()
            df['macro_region'] = df['country_clean'].map(COUNTRY_TO_MACRO_REGION)
            df = df.dropna(subset=['macro_region'])
        group_total = df.groupby(group_col).size()
        group_missing = df[df['amount_usd'] == 0].groupby(group_col).size()
        group_pct = (group_missing / group_total * 100).dropna().sort_values(ascending=False)
        # Filter to groups with enough events
        large_groups = group_total[group_total >= 20]
        group_pct_filtered = group_pct[group_pct.index.isin(large_groups.index)]
        if len(group_pct_filtered) > 0:
            worst = group_pct_filtered.index[0]
            best = group_pct_filtered.index[-1]
            lines.append(f'- Highest missing rate: {worst} ({fmt_pct(group_pct_filtered[worst])})')
            lines.append(f'- Lowest missing rate: {best} ({fmt_pct(group_pct_filtered[best])})')
    elif group_col == 'overview_pie':
        lines.append(f'- Disclosed: {total_events - missing:,} ({fmt_pct(100 - overall_pct)})')
        lines.append(f'- Undisclosed: {missing:,} ({fmt_pct(overall_pct)})')

    return lines


def summarize_investment_type_overview(df, metric, scope_filter, scope_name, start_year):
    """Summary for investment type overview bar charts."""
    filtered = scope_filter(df) if scope_filter else df
    filtered = filtered[filtered['year'] >= start_year]
    if len(filtered) == 0:
        return None

    max_year = int(filtered['year'].max())

    if metric == 'amount':
        type_totals = filtered.groupby('investment_type')['amount_usd'].sum().sort_values(ascending=False)
        total = type_totals.sum()
        top1 = type_totals.index[0]
        top1_val = type_totals.iloc[0]
        top3_share = type_totals.head(3).sum() / total * 100

        lines = [
            f'Chart: investment_type_overview (amount)',
            f'Scope: {scope_name}',
            f'Period: {start_year}-{max_year}',
            '',
            f'- Total disclosed funding: {fmt_usd(total)}',
            f'- Number of investment types: {len(type_totals)}',
            f'- Top type: {top1} ({fmt_usd(top1_val)}, {fmt_pct(top1_val/total*100)} of total)',
        ]
        for i, (typ, val) in enumerate(type_totals.head(5).items(), 1):
            lines.append(f'- #{i}: {typ} — {fmt_usd(val)} ({fmt_pct(val/total*100)})')
        lines.append(f'- Top-3 concentration: {fmt_pct(top3_share)}')

        shares = (type_totals / total * 100).values
        h = hhi(shares)
        lines.append(f'- HHI: {h:.0f} ({hhi_label(h)})')

    else:  # count
        type_totals = filtered.groupby('investment_type').size().sort_values(ascending=False)
        total = type_totals.sum()
        top1 = type_totals.index[0]
        top1_val = type_totals.iloc[0]
        top3_share = type_totals.head(3).sum() / total * 100

        lines = [
            f'Chart: investment_type_overview (count)',
            f'Scope: {scope_name}',
            f'Period: {start_year}-{max_year}',
            '',
            f'- Total funding events: {total:,}',
            f'- Number of investment types: {len(type_totals)}',
            f'- Top type: {top1} ({top1_val:,} events, {fmt_pct(top1_val/total*100)} of total)',
        ]
        for i, (typ, val) in enumerate(type_totals.head(5).items(), 1):
            lines.append(f'- #{i}: {typ} — {val:,} events ({fmt_pct(val/total*100)})')
        lines.append(f'- Top-3 concentration: {fmt_pct(top3_share)}')

        shares = (type_totals / total * 100).values
        h = hhi(shares)
        lines.append(f'- HHI: {h:.0f} ({hhi_label(h)})')

    return lines


def summarize_investment_type_yearly(df, metric, scope_filter, scope_name, start_year):
    """Summary for investment type yearly stacked charts."""
    filtered = scope_filter(df) if scope_filter else df
    filtered = filtered[filtered['year'] >= start_year]
    if len(filtered) == 0:
        return None

    max_year = int(filtered['year'].max())

    if metric == 'amount':
        pivot = filtered.groupby(['year', 'investment_type'])['amount_usd'].sum().unstack(fill_value=0)
    else:
        pivot = filtered.groupby(['year', 'investment_type']).size().unstack(fill_value=0)

    lines = [
        f'Chart: investment_type_yearly ({metric})',
        f'Scope: {scope_name}',
        f'Period: {start_year}-{max_year}',
        '',
    ]

    # Dominant type per recent year
    recent_years = sorted(pivot.index)[-3:]
    for y in recent_years:
        row = pivot.loc[y]
        dominant = row.idxmax()
        dominant_val = row.max()
        total = row.sum()
        pct = dominant_val / total * 100 if total > 0 else 0
        if metric == 'amount':
            lines.append(f'- {y}: dominant type = {dominant} ({fmt_usd(dominant_val)}, {fmt_pct(pct)})')
        else:
            lines.append(f'- {y}: dominant type = {dominant} ({int(dominant_val):,} events, {fmt_pct(pct)})')

    # Growth trends
    if len(pivot) >= 3:
        first_year = pivot.index[0]
        last_year = pivot.index[-1]
        for typ in pivot.columns[:5]:
            start_v = pivot.loc[first_year, typ]
            end_v = pivot.loc[last_year, typ]
            if start_v > 0 and end_v > 0:
                growth = (end_v - start_v) / start_v * 100
                lines.append(f'- {typ}: {growth:+.0f}% change ({first_year} to {last_year})')

    return lines


def summarize_macro_region(df, metric, scope_name, start_year):
    """Summary for macro-region overview and yearly charts."""
    filtered = df[df['year'] >= start_year].copy()
    filtered['macro_region'] = filtered['country_clean'].map(COUNTRY_TO_MACRO_REGION)
    filtered = filtered.dropna(subset=['macro_region'])
    if len(filtered) == 0:
        return None

    max_year = int(filtered['year'].max())
    region_order = ['Americas', 'Europe/Africa', 'Asia/Pacific']

    if metric == 'amount':
        totals = filtered.groupby('macro_region')['amount_usd'].sum()
        grand_total = totals.sum()

        lines = [
            f'Chart: macro_region ({metric})',
            f'Scope: {scope_name}',
            f'Period: {start_year}-{max_year}',
            '',
            f'- Total disclosed funding: {fmt_usd(grand_total)}',
        ]
        for region in region_order:
            val = totals.get(region, 0)
            pct = val / grand_total * 100 if grand_total > 0 else 0
            lines.append(f'- {region}: {fmt_usd(val)} ({fmt_pct(pct)})')
        largest = totals.idxmax()
        lines.append(f'- Largest region: {largest}')

    else:  # count
        totals = filtered.groupby('macro_region').size()
        grand_total = totals.sum()

        lines = [
            f'Chart: macro_region ({metric})',
            f'Scope: {scope_name}',
            f'Period: {start_year}-{max_year}',
            '',
            f'- Total funding events: {grand_total:,}',
        ]
        for region in region_order:
            val = totals.get(region, 0)
            pct = val / grand_total * 100 if grand_total > 0 else 0
            lines.append(f'- {region}: {val:,} events ({fmt_pct(pct)})')
        largest = totals.idxmax()
        lines.append(f'- Largest region: {largest}')

    # Growth trends by region
    yearly_by_region = filtered.groupby(['year', 'macro_region'])
    if metric == 'amount':
        yearly_pivot = yearly_by_region['amount_usd'].sum().unstack(fill_value=0)
    else:
        yearly_pivot = yearly_by_region.size().unstack(fill_value=0)

    if len(yearly_pivot) >= 3:
        first_year = yearly_pivot.index[0]
        last_year = yearly_pivot.index[-1]
        for region in region_order:
            if region in yearly_pivot.columns:
                sv = yearly_pivot.loc[first_year, region]
                ev = yearly_pivot.loc[last_year, region]
                if sv > 0:
                    growth = (ev - sv) / sv * 100
                    lines.append(f'- {region} growth ({first_year}-{last_year}): {growth:+.0f}%')

    return lines


def summarize_category_overview(analysis_df, all_events_df, scope_filter, scope_name, start_year):
    """Summary for category overview (grouped bar) charts."""
    adf = scope_filter(analysis_df) if scope_filter else analysis_df
    edf = scope_filter(all_events_df) if scope_filter else all_events_df
    adf = adf[adf['year'] >= start_year]
    edf = edf[edf['year'] >= start_year]
    if len(adf) == 0:
        return None

    max_year = int(adf['year'].max())

    cat_amount = adf.groupby('category_clean')['amount_usd'].sum().sort_values(ascending=False)
    cat_count = edf.groupby('category_clean').size().sort_values(ascending=False)
    total_amount = cat_amount.sum()
    total_count = cat_count.sum()

    lines = [
        f'Chart: category_overview',
        f'Scope: {scope_name}',
        f'Period: {start_year}-{max_year}',
        '',
        f'- Total disclosed funding: {fmt_usd(total_amount)}',
        f'- Total funding events: {total_count:,}',
        f'- Number of categories: {len(cat_amount)}',
        '',
        f'By funding amount:',
    ]
    for i, (cat, val) in enumerate(cat_amount.head(5).items(), 1):
        pct = val / total_amount * 100
        lines.append(f'- #{i}: {cat} — {fmt_usd(val)} ({fmt_pct(pct)})')

    lines.append('')
    lines.append('By event count:')
    for i, (cat, val) in enumerate(cat_count.head(5).items(), 1):
        pct = val / total_count * 100
        lines.append(f'- #{i}: {cat} — {val:,} events ({fmt_pct(pct)})')

    # Largest gap between amount rank and count rank
    amount_rank = {cat: i for i, cat in enumerate(cat_amount.index)}
    count_rank = {cat: i for i, cat in enumerate(cat_count.index)}
    all_cats = set(amount_rank.keys()) & set(count_rank.keys())
    if all_cats:
        rank_diffs = {cat: abs(amount_rank[cat] - count_rank[cat]) for cat in all_cats}
        biggest_diff_cat = max(rank_diffs, key=rank_diffs.get)
        if rank_diffs[biggest_diff_cat] > 0:
            lines.append('')
            lines.append(f'- Biggest rank gap: {biggest_diff_cat} (#{amount_rank[biggest_diff_cat]+1} by amount, #{count_rank[biggest_diff_cat]+1} by count)')

    return lines


def summarize_category_overview_percentage(analysis_df, all_events_df, scope_filter, scope_name, start_year):
    """Summary for category overview percentage charts."""
    adf = scope_filter(analysis_df) if scope_filter else analysis_df
    edf = scope_filter(all_events_df) if scope_filter else all_events_df
    adf = adf[adf['year'] >= start_year]
    edf = edf[edf['year'] >= start_year]
    if len(adf) == 0:
        return None

    max_year = int(adf['year'].max())

    cat_amount = adf.groupby('category_clean')['amount_usd'].sum().sort_values(ascending=False)
    cat_count = edf.groupby('category_clean').size().sort_values(ascending=False)
    total_amount = cat_amount.sum()
    total_count = cat_count.sum()

    # Percentage shares
    amount_pct = (cat_amount / total_amount * 100) if total_amount > 0 else cat_amount * 0
    count_pct = (cat_count / total_count * 100) if total_count > 0 else cat_count * 0

    lines = [
        f'Chart: category_overview_percentage',
        f'Scope: {scope_name}',
        f'Period: {start_year}-{max_year}',
        '',
        f'- Total disclosed funding: {fmt_usd(total_amount)}',
        f'- Total funding events: {total_count:,}',
        f'- Number of categories: {len(cat_amount)}',
        '',
        'By funding amount share:',
    ]
    for i, (cat, pct) in enumerate(amount_pct.head(5).items(), 1):
        lines.append(f'- #{i}: {cat} — {fmt_pct(pct)} ({fmt_usd(cat_amount[cat])})')

    lines.append('')
    lines.append('By event count share:')
    for i, (cat, pct) in enumerate(count_pct.head(5).items(), 1):
        lines.append(f'- #{i}: {cat} — {fmt_pct(pct)} ({int(cat_count[cat]):,} events)')

    # Capital intensity: categories with biggest gap between funding % and event %
    all_cats = set(amount_pct.index) & set(count_pct.index)
    if all_cats:
        gaps = {cat: amount_pct.get(cat, 0) - count_pct.get(cat, 0) for cat in all_cats}
        most_capital = max(gaps, key=gaps.get)
        most_fragmented = min(gaps, key=gaps.get)
        lines.append('')
        if gaps[most_capital] > 0:
            lines.append(f'- Most capital-intensive: {most_capital} (funding {fmt_pct(amount_pct[most_capital])} vs events {fmt_pct(count_pct[most_capital])})')
        if gaps[most_fragmented] < 0:
            lines.append(f'- Most fragmented: {most_fragmented} (funding {fmt_pct(amount_pct[most_fragmented])} vs events {fmt_pct(count_pct[most_fragmented])})')

    # HHI for both distributions
    h_amount = hhi(amount_pct.values)
    h_count = hhi(count_pct.values)
    lines.append(f'- HHI funding: {h_amount:.0f} ({hhi_label(h_amount)})')
    lines.append(f'- HHI events: {h_count:.0f} ({hhi_label(h_count)})')

    return lines


def summarize_category_pie(df, metric, scope_filter, scope_name, start_year):
    """Summary for category FA/FE pie charts."""
    filtered = scope_filter(df) if scope_filter else df
    filtered = filtered[filtered['year'] >= start_year]
    if len(filtered) == 0:
        return None

    max_year = int(filtered['year'].max())

    if metric == 'amount':
        cat_totals = filtered.groupby('category_clean')['amount_usd'].sum().sort_values(ascending=False)
        total = cat_totals.sum()
        label = 'funding amount'
    else:
        cat_totals = filtered.groupby('category_clean').size().sort_values(ascending=False)
        total = cat_totals.sum()
        label = 'event count'

    lines = [
        f'Chart: category_pie ({metric})',
        f'Scope: {scope_name}',
        f'Period: {start_year}-{max_year}',
        '',
    ]

    if metric == 'amount':
        lines.append(f'- Total {label}: {fmt_usd(total)}')
    else:
        lines.append(f'- Total {label}: {total:,}')

    lines.append(f'- Number of categories: {len(cat_totals)}')

    shares = (cat_totals / total * 100)
    for i, (cat, pct) in enumerate(shares.head(5).items(), 1):
        if metric == 'amount':
            lines.append(f'- #{i}: {cat} — {fmt_pct(pct)} ({fmt_usd(cat_totals[cat])})')
        else:
            lines.append(f'- #{i}: {cat} — {fmt_pct(pct)} ({int(cat_totals[cat]):,} events)')

    top3_share = shares.head(3).sum()
    lines.append(f'- Top-3 concentration: {fmt_pct(top3_share)}')

    h = hhi(shares.values)
    lines.append(f'- HHI: {h:.0f} ({hhi_label(h)})')

    return lines


def summarize_funding_by_category_pie(df, scope_filter, scope_name, start_year):
    """Summary for funding_by_category_pie charts (market segment pies)."""
    filtered = scope_filter(df) if scope_filter else df
    filtered = filtered[filtered['year'] >= start_year]
    if len(filtered) == 0:
        return None

    max_year = int(filtered['year'].max())

    cat_totals = filtered.groupby('category_clean')['amount_usd'].sum().sort_values(ascending=False)
    total = cat_totals.sum()

    lines = [
        f'Chart: funding_by_category_pie',
        f'Scope: {scope_name}',
        f'Period: {start_year}-{max_year}',
        '',
        f'- Total disclosed funding: {fmt_usd(total)}',
        f'- Number of categories: {len(cat_totals)}',
    ]

    shares = (cat_totals / total * 100)
    for i, (cat, pct) in enumerate(shares.head(5).items(), 1):
        lines.append(f'- #{i}: {cat} — {fmt_pct(pct)} ({fmt_usd(cat_totals[cat])})')

    top3_share = shares.head(3).sum()
    lines.append(f'- Top-3 concentration: {fmt_pct(top3_share)}')

    h = hhi(shares.values)
    lines.append(f'- HHI: {h:.0f} ({hhi_label(h)})')

    return lines


def summarize_funding_by_category_yearly(df, scope_filter, scope_name, start_year):
    """Summary for funding_by_category_yearly stacked bar charts."""
    filtered = scope_filter(df) if scope_filter else df
    filtered = filtered[filtered['year'] >= start_year]
    if len(filtered) == 0:
        return None

    max_year = int(filtered['year'].max())

    pivot = filtered.groupby(['year', 'category_clean'])['amount_usd'].sum().unstack(fill_value=0)
    total = filtered['amount_usd'].sum()

    lines = [
        f'Chart: funding_by_category_yearly',
        f'Scope: {scope_name}',
        f'Period: {start_year}-{max_year}',
        '',
        f'- Total disclosed funding: {fmt_usd(total)}',
        f'- Number of categories: {len(pivot.columns)}',
    ]

    # Dominant category per recent year
    recent_years = sorted(pivot.index)[-3:]
    for y in recent_years:
        row = pivot.loc[y]
        dominant = row.idxmax()
        dominant_val = row.max()
        row_total = row.sum()
        pct = dominant_val / row_total * 100 if row_total > 0 else 0
        lines.append(f'- {y}: dominant = {dominant} ({fmt_usd(dominant_val)}, {fmt_pct(pct)})')

    # Emerging/declining categories
    if len(pivot) >= 6:
        early = pivot.head(3).sum()
        late = pivot.tail(3).sum()
        early_total = early.sum()
        late_total = late.sum()
        if early_total > 0 and late_total > 0:
            early_shares = early / early_total * 100
            late_shares = late / late_total * 100
            share_change = late_shares - early_shares
            gainers = share_change.sort_values(ascending=False).head(2)
            losers = share_change.sort_values().head(2)
            lines.append('')
            lines.append('Share-shift (first 3 yrs vs last 3 yrs):')
            for cat, change in gainers.items():
                lines.append(f'- Emerging: {cat} ({change:+.1f} pp)')
            for cat, change in losers.items():
                lines.append(f'- Declining: {cat} ({change:+.1f} pp)')

    return lines


def summarize_funded_percentage(companies_df, scope_filter, scope_name):
    """Summary for funded percentage donut charts."""
    df = scope_filter(companies_df) if scope_filter else companies_df
    if len(df) == 0:
        return None

    total = len(df)
    funded = is_yes(df['Is Funded']).sum()
    unfunded = total - funded
    pct = funded / total * 100

    lines = [
        f'Chart: funded_percentage',
        f'Scope: {scope_name}',
        '',
        f'- Total companies: {total:,}',
        f'- Funded: {funded:,} ({fmt_pct(pct)})',
        f'- Not funded: {unfunded:,} ({fmt_pct(100-pct)})',
    ]

    return lines


def summarize_companies_by_funding_type(companies_df, scope_filter, scope_name):
    """Summary for companies by funding type (Company Stage) horizontal bar charts."""
    df = scope_filter(companies_df) if scope_filter else companies_df
    if len(df) == 0:
        return None

    # Find stage column
    stage_col = None
    for candidate in ['Company Stage', 'Company Stage_Companies']:
        if candidate in df.columns:
            stage_col = candidate
            break
    if stage_col is None:
        return None

    stage_counts = df[stage_col].value_counts()
    total = stage_counts.sum()

    lines = [
        f'Chart: companies_by_funding_type',
        f'Scope: {scope_name}',
        '',
        f'- Total companies: {total:,}',
        f'- Number of stages: {len(stage_counts)}',
    ]

    for i, (stage, count) in enumerate(stage_counts.head(5).items(), 1):
        pct = count / total * 100
        lines.append(f'- #{i}: {stage} — {count:,} ({fmt_pct(pct)})')

    top3_share = stage_counts.head(3).sum() / total * 100
    lines.append(f'- Top-3 stage concentration: {fmt_pct(top3_share)}')

    return lines


def summarize_companies_by_year_deadpool(companies_df, scope_filter, scope_name, start_year):
    """Summary for companies by year + deadpool stacked bar charts."""
    df = scope_filter(companies_df) if scope_filter else companies_df
    df = df[df['founded_year'] >= start_year]
    if len(df) == 0:
        return None

    total = len(df)
    dead = is_yes(df['Is Deadpooled']).sum()
    dead_pct = dead / total * 100
    max_year = int(df['founded_year'].max())

    # Deadpool by year
    df_copy = df.copy()
    df_copy['is_dead'] = is_yes(df_copy['Is Deadpooled'])
    yearly_dead = df_copy.groupby('founded_year')['is_dead'].mean() * 100

    # Filter to years with meaningful sample
    yearly_counts = df_copy.groupby('founded_year').size()
    meaningful = yearly_counts[yearly_counts >= 5].index
    yearly_dead_filtered = yearly_dead[yearly_dead.index.isin(meaningful)]

    lines = [
        f'Chart: companies_by_year_deadpool',
        f'Scope: {scope_name}',
        f'Period: {start_year}-{max_year}',
        '',
        f'- Total companies: {total:,}',
        f'- Deadpooled: {dead:,} ({fmt_pct(dead_pct)})',
        f'- Active: {total-dead:,} ({fmt_pct(100-dead_pct)})',
    ]

    if len(yearly_dead_filtered) > 0:
        worst_year = int(yearly_dead_filtered.idxmax())
        worst_pct = yearly_dead_filtered.max()
        lines.append(f'- Highest deadpool rate year: {worst_year} ({fmt_pct(worst_pct)})')

        # Trend
        recent = sorted(yearly_dead_filtered.index)[-3:]
        if len(recent) >= 2:
            vals = [yearly_dead_filtered[y] for y in recent]
            direction = 'increasing' if vals[-1] > vals[-2] else 'decreasing'
            lines.append(f'- Recent deadpool trend: {direction}')

    return lines


def summarize_regional_company_bars(companies_df, metric, scope_name):
    """Summary for regional company bar charts (count, deadpooled, funded, pct_funded)."""
    df = companies_df.copy()
    df['macro_region'] = df['country_clean'].map(COUNTRY_TO_MACRO_REGION)
    df = df.dropna(subset=['macro_region'])
    if len(df) == 0:
        return None

    region_order = ['Americas', 'Europe/Africa', 'Asia/Pacific']

    lines = [
        f'Chart: regional_companies_{metric}',
        f'Scope: {scope_name}',
        '',
    ]

    if metric == 'count':
        totals = df.groupby('macro_region').size()
        grand = totals.sum()
        lines.append(f'- Total companies: {grand:,}')
        for region in region_order:
            val = totals.get(region, 0)
            pct = val / grand * 100
            lines.append(f'- {region}: {val:,} ({fmt_pct(pct)})')
    elif metric == 'deadpooled':
        for region in region_order:
            rdf = df[df['macro_region'] == region]
            total_r = len(rdf)
            dead = is_yes(rdf['Is Deadpooled']).sum()
            pct = dead / total_r * 100 if total_r > 0 else 0
            lines.append(f'- {region}: {dead:,}/{total_r:,} deadpooled ({fmt_pct(pct)})')
    elif metric == 'funded':
        for region in region_order:
            rdf = df[df['macro_region'] == region]
            total_r = len(rdf)
            funded = is_yes(rdf['Is Funded']).sum()
            pct = funded / total_r * 100 if total_r > 0 else 0
            lines.append(f'- {region}: {funded:,}/{total_r:,} funded ({fmt_pct(pct)})')
    elif metric == 'pct_funded':
        pcts = {}
        for region in region_order:
            rdf = df[df['macro_region'] == region]
            total_r = len(rdf)
            funded = is_yes(rdf['Is Funded']).sum()
            pct = funded / total_r * 100 if total_r > 0 else 0
            pcts[region] = pct
            lines.append(f'- {region}: {fmt_pct(pct)} funded ({funded:,}/{total_r:,})')
        # Ratio between highest and lowest
        if pcts:
            highest = max(pcts, key=pcts.get)
            lowest = min(pcts, key=pcts.get)
            if pcts[lowest] > 0:
                ratio = pcts[highest] / pcts[lowest]
                lines.append(f'- Ratio {highest} vs {lowest}: {ratio:.1f}x')

    return lines


def summarize_category_status(companies_df, metric, scope_name):
    """Summary for category status (deadpooled/funded) stacked percentage charts."""
    df = companies_df.copy()
    if len(df) == 0:
        return None

    if metric == 'deadpooled':
        df['status'] = is_yes(df['Is Deadpooled'])
        label = 'deadpooled'
    elif metric == 'funded':
        df['status'] = is_yes(df['Is Funded'])
        label = 'funded'
    else:
        return None

    cat_total = df.groupby('category_clean').size()
    cat_yes = df[df['status']].groupby('category_clean').size()
    cat_pct = (cat_yes / cat_total * 100).dropna().sort_values(ascending=False)

    # Filter to categories with enough companies
    large_cats = cat_total[cat_total >= 10].index
    cat_pct_filtered = cat_pct[cat_pct.index.isin(large_cats)]

    lines = [
        f'Chart: category_{label}_status',
        f'Scope: {scope_name}',
        '',
        f'- Total companies: {len(df):,}',
        f'- Overall {label} rate: {fmt_pct(df["status"].mean() * 100)}',
    ]

    if len(cat_pct_filtered) > 0:
        highest_cat = cat_pct_filtered.index[0]
        lowest_cat = cat_pct_filtered.index[-1]
        lines.append(f'- Highest {label} rate: {highest_cat} ({fmt_pct(cat_pct_filtered[highest_cat])})')
        lines.append(f'- Lowest {label} rate: {lowest_cat} ({fmt_pct(cat_pct_filtered[lowest_cat])})')
        lines.append(f'- Range: {fmt_pct(cat_pct_filtered.max() - cat_pct_filtered.min())} spread')

        # Top 3
        lines.append('')
        for i, (cat, pct) in enumerate(cat_pct_filtered.head(3).items(), 1):
            count = int(cat_total[cat])
            lines.append(f'- #{i} highest: {cat} — {fmt_pct(pct)} ({count:,} companies)')

    return lines


def summarize_top10_countries(df, metric_col, scope_name):
    """Summary for top 10 country charts."""
    if metric_col == 'count':
        totals = df.groupby('country_clean').size().sort_values(ascending=False)
    elif metric_col == 'sum_amount':
        totals = df.groupby('country_clean')['amount_usd'].sum().sort_values(ascending=False)
    else:
        totals = df.groupby('country_clean')[metric_col].sum().sort_values(ascending=False)

    top10 = totals.head(10)
    if len(top10) == 0:
        return None

    grand_total = totals.sum()
    top1 = top10.index[0]
    top1_val = top10.iloc[0]
    top10_share = top10.sum() / grand_total * 100 if grand_total > 0 else 0

    lines = [
        f'Chart: top10_countries',
        f'Scope: {scope_name}',
        '',
        f'- #1 country: {top1} ({top1_val:,.0f})',
        f'- Top-10 concentration: {fmt_pct(top10_share)} of total',
    ]

    for i, (country, val) in enumerate(top10.items(), 1):
        pct = val / grand_total * 100 if grand_total > 0 else 0
        lines.append(f'- #{i}: {country} — {val:,.0f} ({fmt_pct(pct)})')

    # HHI of top 10
    shares = (top10 / grand_total * 100).values
    h = hhi(shares)
    lines.append(f'- HHI (top 10): {h:.0f} ({hhi_label(h)})')

    return lines


# --- Missing Charts notebook summaries ---

def summarize_companies_founded_buckets(companies_df, scope_filter, scope_name):
    """Summary for bucketed companies-founded-by-year bar charts (Missing Charts)."""
    df = scope_filter(companies_df) if scope_filter else companies_df
    if len(df) == 0:
        return None

    total = len(df)
    pre_2000 = len(df[df['founded_year'] < 2000])
    y2000_2010 = len(df[(df['founded_year'] >= 2000) & (df['founded_year'] <= 2010)])
    post_2010 = len(df[df['founded_year'] > 2010])

    yearly = df[df['founded_year'] > 2010].groupby('founded_year').size()
    peak_year = int(yearly.idxmax()) if len(yearly) > 0 else None
    peak_count = int(yearly.max()) if len(yearly) > 0 else 0

    lines = [
        f'Chart: companies_founded_by_year (bucketed)',
        f'Scope: {scope_name}',
        '',
        f'- Total companies: {total:,}',
        f'- Pre-2000: {pre_2000:,} ({fmt_pct(pre_2000/total*100)})',
        f'- 2000-2010: {y2000_2010:,} ({fmt_pct(y2000_2010/total*100)})',
        f'- 2011+: {post_2010:,} ({fmt_pct(post_2010/total*100)})',
    ]
    if peak_year:
        lines.append(f'- Peak year (2011+): {peak_year} ({peak_count:,} companies)')

    return lines


def summarize_companies_by_year_funded_stacked(companies_df, scope_filter, scope_name):
    """Summary for stacked funded Yes/No by year charts (Missing Charts)."""
    df = scope_filter(companies_df) if scope_filter else companies_df
    if len(df) == 0:
        return None

    total = len(df)
    funded = is_yes(df['Is Funded']).sum()
    pct_funded = funded / total * 100

    lines = [
        f'Chart: companies_by_year_funded_stacked',
        f'Scope: {scope_name}',
        '',
        f'- Total companies: {total:,}',
        f'- Funded: {funded:,} ({fmt_pct(pct_funded)})',
        f'- Not funded: {total-funded:,} ({fmt_pct(100-pct_funded)})',
    ]

    # Funded rate by period
    for label, mask in [('Pre-2000', df['founded_year'] < 2000),
                        ('2000-2010', (df['founded_year'] >= 2000) & (df['founded_year'] <= 2010)),
                        ('2011+', df['founded_year'] > 2010)]:
        subset = df[mask]
        if len(subset) > 0:
            sub_funded = is_yes(subset['Is Funded']).sum()
            sub_pct = sub_funded / len(subset) * 100
            lines.append(f'- {label}: {fmt_pct(sub_pct)} funded ({sub_funded:,}/{len(subset):,})')

    return lines


def summarize_regional_status_bars(companies_df, metric, scope_name):
    """Summary for regional status percentage bars (Missing Charts)."""
    df = companies_df.copy()
    df['macro_region'] = df['country_clean'].map(COUNTRY_TO_MACRO_REGION)
    df = df.dropna(subset=['macro_region'])
    if len(df) == 0:
        return None

    region_order = ['Americas', 'Europe/Africa', 'Asia/Pacific']

    if metric == 'deadpooled':
        df['status'] = is_yes(df['Is Deadpooled'])
        label = 'deadpooled'
    elif metric == 'funded':
        df['status'] = is_yes(df['Is Funded'])
        label = 'funded'
    else:
        return None

    lines = [
        f'Chart: regional_{label}_percentage',
        f'Scope: {scope_name}',
        '',
    ]

    for region in region_order:
        rdf = df[df['macro_region'] == region]
        total_r = len(rdf)
        yes_r = rdf['status'].sum()
        pct = yes_r / total_r * 100 if total_r > 0 else 0
        lines.append(f'- {region}: {fmt_pct(pct)} {label} ({yes_r:,}/{total_r:,})')

    return lines


def summarize_funded_region_pie(companies_df, scope_name):
    """Summary for funded companies regional pie chart (Missing Charts)."""
    df = companies_df.copy()
    df = df[is_yes(df['Is Funded'])]
    df['macro_region'] = df['country_clean'].map(COUNTRY_TO_MACRO_REGION)
    df = df.dropna(subset=['macro_region'])
    if len(df) == 0:
        return None

    region_order = ['Americas', 'Europe/Africa', 'Asia/Pacific']
    counts = df.groupby('macro_region').size().reindex(region_order, fill_value=0)
    total = counts.sum()

    lines = [
        f'Chart: regional_funded_companies_pie',
        f'Scope: {scope_name}',
        '',
        f'- Total funded companies: {total:,}',
    ]

    for region in region_order:
        val = counts[region]
        pct = val / total * 100 if total > 0 else 0
        lines.append(f'- {region}: {val:,} ({fmt_pct(pct)})')

    return lines


def summarize_global_vc(vc_df, start_year, scope_name):
    """Summary for global VC totals chart."""
    df = vc_df[vc_df['Year'] >= start_year].copy()
    if len(df) == 0:
        return None

    total = df['Global VC Investment (USD billions)'].sum()
    peak_year = int(df.loc[df['Global VC Investment (USD billions)'].idxmax(), 'Year'])
    peak_val = df['Global VC Investment (USD billions)'].max()
    max_year = int(df['Year'].max())

    lines = [
        f'Chart: global_vc_totals',
        f'Scope: {scope_name}',
        f'Period: {start_year}-{max_year}',
        '',
        f'- Total VC investment: ${total:.0f}B',
        f'- Peak year: {peak_year} (${peak_val:.0f}B)',
    ]

    # Recent trend
    recent = df.sort_values('Year').tail(3)
    for _, row in recent.iterrows():
        lines.append(f'- {int(row["Year"])}: ${row["Global VC Investment (USD billions)"]:.0f}B')

    return lines

In [ ]:
# Cell 6: Master Summary Generation Loop

# Missing Charts notebook uses FUNDED_START_YEAR (not in main notebook config)
FUNDED_START_YEAR = 2011

os.makedirs(CHART_OUTPUT_DIR, exist_ok=True)
summaries_generated = []
summaries_failed = []

def generate_summary(chart_name, summary_fn, *args, **kwargs):
    """Wrapper: call summary function, save output, track success/failure."""
    try:
        lines = summary_fn(*args, **kwargs)
        if lines is not None:
            save_summary(chart_name, lines)
            summaries_generated.append(chart_name)
        else:
            print(f'  SKIP {chart_name}: no data')
    except Exception as e:
        summaries_failed.append((chart_name, str(e)))
        print(f'  ERROR {chart_name}: {e}')

if SKIP_TRACXN_UPLOAD:
    print("Skipping Tracxn summary generation (SKIP_TRACXN_UPLOAD = True).\n")
else:
    # --- Scope Filter Definitions ---
    americas_countries = [c for st in MACRO_REGION_GROUPS['Americas'] for c in SECOND_TIER_REGIONS[st]]
    asia_pacific_countries = [c for st in MACRO_REGION_GROUPS['Asia/Pacific'] for c in SECOND_TIER_REGIONS[st]]
    europe_africa_countries = [c for st in MACRO_REGION_GROUPS['Europe/Africa'] for c in SECOND_TIER_REGIONS[st]]

    scope_americas = lambda df: df[df['country_clean'].isin(americas_countries)]
    scope_asia_pacific = lambda df: df[df['country_clean'].isin(asia_pacific_countries)]
    scope_europe_africa = lambda df: df[df['country_clean'].isin(europe_africa_countries)]
    scope_usa = lambda df: df[df['country_clean'] == 'United States']

    # =====================================================================
    # MAIN NOTEBOOK CHARTS (from Robotics_Industry_Analysis.ipynb Cell 7)
    # =====================================================================

    # === 1. Companies Founded Charts (7 charts) ===
    print("Generating companies-founded summaries...")
    count_before = len(summaries_generated)

    generate_summary('companies_founded_absolute_1900',
        summarize_companies_founded, companies_df, None, 'Global', 1900)

    generate_summary('companies_founded_absolute_all_years',
        summarize_companies_founded, companies_df, None, 'Global', 1900)

    generate_summary('companies_founded_absolute_2000',
        summarize_companies_founded, companies_df, None, 'Global', 2000)

    generate_summary('companies_founded_bar_2000',
        summarize_companies_founded, companies_df, None, 'Global', 2000)

    generate_summary('companies_founded_outcomes_absolute',
        summarize_companies_founded, companies_df, None, 'Global', 2000)

    generate_summary('companies_founded_by_sector_stacked_2000',
        summarize_companies_founded, companies_df, None, 'Global', 2000)

    generate_summary('companies_founded_by_funding_status_stacked_2000',
        summarize_companies_founded, companies_df, None, 'Global', 2000)

    print(f"  Companies-founded summaries: {len(summaries_generated) - count_before}")

    # === 2. Global Funding Charts (2 charts) ===
    print("\nGenerating global funding summaries...")
    count_before = len(summaries_generated)

    generate_summary('total_global_funding_by_year_2000',
        summarize_total_funding, analysis_df, 'Global', 2000)

    generate_summary('robotics_funding_by_year_2000_onwards',
        summarize_stacked_funding, analysis_df, all_events_df, None, 'Global', 2000)

    print(f"  Global funding summaries: {len(summaries_generated) - count_before}")

    # === 3. Sector Charts ===
    print("\nGenerating sector summaries...")
    count_before = len(summaries_generated)
    sectors = sorted(analysis_df['category_clean'].unique())

    for sector in sectors:
        sanitized = sanitize_filename(sector)
        filename = f'funding_by_round_size_{sanitized}_2000_onwards'
        filter_fn = lambda df, s=sector: df[df['category_clean'] == s]
        generate_summary(filename,
            summarize_stacked_funding, analysis_df, all_events_df, filter_fn, sector, 2000)

    print(f"  Sector summaries: {len(summaries_generated) - count_before}")

    # === 4. Region Charts (12 charts) ===
    print("\nGenerating region summaries...")
    count_before = len(summaries_generated)

    for region, countries in SECOND_TIER_REGIONS.items():
        sanitized = sanitize_filename(region)
        filename = f'funding_by_round_size_{sanitized}_2000_onwards'
        filter_fn = lambda df, c=countries: df[df['country_clean'].isin(c)]
        generate_summary(filename,
            summarize_stacked_funding, analysis_df, all_events_df, filter_fn, region, 2000)

    print(f"  Region summaries: {len(summaries_generated) - count_before}")

    # === 5. Subcategory Charts ===
    print("\nGenerating subcategory summaries...")
    count_before = len(summaries_generated)

    subcat_col = 'Subcategory_Companies' if 'Subcategory_Companies' in analysis_df.columns else 'Subcategory'
    subcategories = sorted(analysis_df[subcat_col].dropna().unique())

    for subcategory in subcategories:
        sanitized = sanitize_filename(subcategory)
        filename = f'funding_by_round_size_{sanitized}_2000_onwards'
        filter_fn = lambda df, s=subcategory, col=subcat_col: df[df[col] == s]
        generate_summary(filename,
            summarize_stacked_funding, analysis_df, all_events_df, filter_fn, subcategory, 2000)

    print(f"  Subcategory summaries: {len(summaries_generated) - count_before}")

    # === 6. Missing Funding Analysis Charts (6 charts) ===
    print("\nGenerating missing funding summaries...")
    count_before = len(summaries_generated)

    generate_summary('missing_funding_by_year',
        summarize_missing_funding, all_events_df, MISSING_FUNDING_START_YEAR, 'year', 'Global')

    generate_summary('missing_funding_by_country',
        summarize_missing_funding, all_events_df, MISSING_FUNDING_START_YEAR, 'country_clean', 'Global')

    generate_summary('missing_funding_by_region',
        summarize_missing_funding, all_events_df, MISSING_FUNDING_START_YEAR, 'macro_region', 'Global')

    generate_summary('missing_funding_by_category',
        summarize_missing_funding, all_events_df, MISSING_FUNDING_START_YEAR, 'category_clean', 'Global')

    generate_summary('missing_funding_by_type',
        summarize_missing_funding, all_events_df, MISSING_FUNDING_START_YEAR, 'investment_type', 'Global')

    generate_summary('missing_funding_overview_pie',
        summarize_missing_funding, all_events_df, MISSING_FUNDING_START_YEAR, 'overview_pie', 'Global')

    print(f"  Missing funding summaries: {len(summaries_generated) - count_before}")

    # === 7. Investment Type Charts ===
    print("\nGenerating investment type summaries...")
    count_before = len(summaries_generated)

    IT_SCOPES = {
        'global': ('Global', None),
        'americas': ('Americas', lambda df: df[df['country_clean'].isin(
            [c for st in MACRO_REGION_GROUPS['Americas'] for c in SECOND_TIER_REGIONS[st]])]),
        'europe_africa': ('Europe/Africa', lambda df: df[df['country_clean'].isin(
            [c for st in MACRO_REGION_GROUPS['Europe/Africa'] for c in SECOND_TIER_REGIONS[st]])]),
        'asia_pacific': ('Asia/Pacific', lambda df: df[df['country_clean'].isin(
            [c for st in MACRO_REGION_GROUPS['Asia/Pacific'] for c in SECOND_TIER_REGIONS[st]])]),
    }

    for scope_key, (scope_name, scope_fn) in IT_SCOPES.items():
        prefix = sanitize_filename(scope_name).lower()

        generate_summary(f'{prefix}_investment_type_FA_overview',
            summarize_investment_type_overview, analysis_df, 'amount', scope_fn, scope_name, INVESTMENT_TYPE_START_YEAR)

        generate_summary(f'{prefix}_investment_type_FA_yearly',
            summarize_investment_type_yearly, analysis_df, 'amount', scope_fn, scope_name, INVESTMENT_TYPE_START_YEAR)

        generate_summary(f'{prefix}_investment_type_FE_overview',
            summarize_investment_type_overview, all_events_df, 'count', scope_fn, scope_name, INVESTMENT_TYPE_START_YEAR)

        generate_summary(f'{prefix}_investment_type_FE_yearly',
            summarize_investment_type_yearly, all_events_df, 'count', scope_fn, scope_name, INVESTMENT_TYPE_START_YEAR)

    # Macro-region overview/yearly charts
    generate_summary('macro_region_FA_overview',
        summarize_macro_region, analysis_df, 'amount', 'Global', INVESTMENT_TYPE_START_YEAR)
    generate_summary('macro_region_FE_overview',
        summarize_macro_region, all_events_df, 'count', 'Global', INVESTMENT_TYPE_START_YEAR)
    generate_summary('macro_region_FA_yearly',
        summarize_macro_region, analysis_df, 'amount', 'Global', INVESTMENT_TYPE_START_YEAR)
    generate_summary('macro_region_FE_yearly',
        summarize_macro_region, all_events_df, 'count', 'Global', INVESTMENT_TYPE_START_YEAR)

    print(f"  Investment type summaries: {len(summaries_generated) - count_before}")

    # === 8. Category Overview Charts ===
    print("\nGenerating category overview summaries...")
    count_before = len(summaries_generated)

    CAT_OV_SCOPES = {
        'global': ('Global', None),
        'americas': ('Americas', lambda df: df[df['country_clean'].isin(
            [c for st in MACRO_REGION_GROUPS['Americas'] for c in SECOND_TIER_REGIONS[st]])]),
        'europe_africa': ('Europe/Africa', lambda df: df[df['country_clean'].isin(
            [c for st in MACRO_REGION_GROUPS['Europe/Africa'] for c in SECOND_TIER_REGIONS[st]])]),
        'asia_pacific': ('Asia/Pacific', lambda df: df[df['country_clean'].isin(
            [c for st in MACRO_REGION_GROUPS['Asia/Pacific'] for c in SECOND_TIER_REGIONS[st]])]),
    }

    for scope_key, (scope_name, scope_fn) in CAT_OV_SCOPES.items():
        prefix = sanitize_filename(scope_name).lower()

        generate_summary(f'{prefix}_category_overview',
            summarize_category_overview, analysis_df, all_events_df, scope_fn, scope_name, INVESTMENT_TYPE_START_YEAR)

    print(f"  Category overview summaries: {len(summaries_generated) - count_before}")


    # === 8b. Category Overview Percentage Charts ===
    print("\nGenerating category overview percentage summaries...")
    count_before = len(summaries_generated)

    for scope_key, (scope_name, scope_fn) in CAT_OV_SCOPES.items():
        prefix = sanitize_filename(scope_name).lower()

        generate_summary(f'{prefix}_category_overview_percentage',
            summarize_category_overview_percentage, analysis_df, all_events_df, scope_fn, scope_name, INVESTMENT_TYPE_START_YEAR)

    print(f"  Category overview percentage summaries: {len(summaries_generated) - count_before}")

    # === 9. Category Pie Charts ===
    print("\nGenerating category pie summaries...")
    count_before = len(summaries_generated)

    for scope_key, (scope_name, scope_fn) in CAT_OV_SCOPES.items():
        prefix = sanitize_filename(scope_name).lower()

        generate_summary(f'{prefix}_category_FA_pie',
            summarize_category_pie, analysis_df, 'amount', scope_fn, scope_name, INVESTMENT_TYPE_START_YEAR)

        generate_summary(f'{prefix}_category_FE_pie',
            summarize_category_pie, all_events_df, 'count', scope_fn, scope_name, INVESTMENT_TYPE_START_YEAR)

    print(f"  Category pie summaries: {len(summaries_generated) - count_before}")

    # === 10. Market Segment Charts ===
    print("\nGenerating market segment summaries...")
    count_before = len(summaries_generated)

    generate_summary('global_funding_by_category_pie',
        summarize_funding_by_category_pie, analysis_df, None, 'Global', INVESTMENT_TYPE_START_YEAR)

    MS_SCOPES = {
        'global': ('Global', None),
        'americas': ('Americas', lambda df: df[df['country_clean'].isin(
            [c for st in MACRO_REGION_GROUPS['Americas'] for c in SECOND_TIER_REGIONS[st]])]),
        'asia_pacific': ('Asia/Pacific', lambda df: df[df['country_clean'].isin(
            [c for st in MACRO_REGION_GROUPS['Asia/Pacific'] for c in SECOND_TIER_REGIONS[st]])]),
        'europe_africa': ('Europe/Africa', lambda df: df[df['country_clean'].isin(
            [c for st in MACRO_REGION_GROUPS['Europe/Africa'] for c in SECOND_TIER_REGIONS[st]])]),
    }

    for scope_key, (scope_name, scope_fn) in MS_SCOPES.items():
        prefix = sanitize_filename(scope_name)

        generate_summary(f'{prefix}_funding_by_category_yearly',
            summarize_funding_by_category_yearly, analysis_df, scope_fn, scope_name, INVESTMENT_TYPE_START_YEAR)

        if scope_fn is not None:
            generate_summary(f'{prefix}_funding_by_category_pie',
                summarize_funding_by_category_pie, analysis_df, scope_fn, scope_name, INVESTMENT_TYPE_START_YEAR)

    print(f"  Market segment summaries: {len(summaries_generated) - count_before}")

    # === 11. State of Market Charts ===
    print("\nGenerating state of market summaries...")
    count_before = len(summaries_generated)

    SOM_SCOPES = {
        'global': ('Global', None),
        'americas': ('Americas', lambda df: df[df['country_clean'].isin(
            [c for st in MACRO_REGION_GROUPS['Americas'] for c in SECOND_TIER_REGIONS[st]])]),
        'asia_pacific': ('Asia/Pacific', lambda df: df[df['country_clean'].isin(
            [c for st in MACRO_REGION_GROUPS['Asia/Pacific'] for c in SECOND_TIER_REGIONS[st]])]),
        'europe_africa': ('Europe/Africa', lambda df: df[df['country_clean'].isin(
            [c for st in MACRO_REGION_GROUPS['Europe/Africa'] for c in SECOND_TIER_REGIONS[st]])]),
        'usa': ('USA', lambda df: df[df['country_clean'] == 'United States']),
    }

    for scope_key, (scope_name, scope_fn) in SOM_SCOPES.items():
        prefix = sanitize_filename(scope_name).lower()

        generate_summary(f'{prefix}_funded_percentage',
            summarize_funded_percentage, companies_df, scope_fn, scope_name)

        generate_summary(f'{prefix}_companies_by_funding_type',
            summarize_companies_by_funding_type, companies_df, scope_fn, scope_name)

        generate_summary(f'{prefix}_companies_by_year_deadpool',
            summarize_companies_by_year_deadpool, companies_df, scope_fn, scope_name, 2000)

    # Regional company bars (global only, 4 variants)
    for metric in ['count', 'deadpooled', 'funded', 'pct_funded']:
        generate_summary(f'regional_companies_{metric}',
            summarize_regional_company_bars, companies_df, metric, 'Global')

    # Category status charts (global only)
    generate_summary('category_deadpooled_status',
        summarize_category_status, companies_df, 'deadpooled', 'Global')

    generate_summary('category_funded_status',
        summarize_category_status, companies_df, 'funded', 'Global')

    print(f"  State of market summaries: {len(summaries_generated) - count_before}")

    # === 12. Top 10 Country Charts ===
    print("\nGenerating top 10 country summaries...")
    count_before = len(summaries_generated)

    stage_col = None
    for candidate in ['Company Stage', 'Company Stage_Companies']:
        if candidate in companies_df.columns:
            stage_col = candidate
            break

    if stage_col is not None:
        stages = [
            'Seed', 'Early Stage', 'Late Stage',
            'Series B,C,D,E,F,G', 'Acquired', 'Acqui-Hired',
            'Deadpooled', 'Public', 'Unfunded', 'Funding Raised', 'Seed+A'
        ]
        for stage in stages:
            safe_stage = sanitize_filename(stage)
            if ',' in stage:
                stage_df = companies_df[companies_df[stage_col].str.contains(
                    '|'.join(stage.split(',')), case=False, na=False)]
            elif stage == 'Seed+A':
                stage_df = companies_df[companies_df[stage_col].str.strip().isin(['Seed', 'Series A'])]
            else:
                stage_df = companies_df[companies_df[stage_col].str.strip().str.lower() == stage.lower()]

            if len(stage_df) > 0:
                generate_summary(f'top10_stage_{safe_stage}',
                    summarize_top10_countries, stage_df, 'count', f'{stage} companies')

    # Total Funding
    generate_summary('top10_total_funding',
        summarize_top10_countries, analysis_df, 'sum_amount', 'Total Funding')

    # Total Employee Count
    emp_col = None
    for candidate in ['Total Employee Count', 'Total Employee Count_Companies']:
        if candidate in companies_df.columns:
            emp_col = candidate
            break

    if emp_col is not None:
        emp_df = companies_df[companies_df[emp_col].notna()].copy()
        emp_df[emp_col] = pd.to_numeric(emp_df[emp_col], errors='coerce')
        emp_df = emp_df[emp_df[emp_col] > 0]
        if len(emp_df) > 0:
            generate_summary('top10_total_employee_count',
                summarize_top10_countries, emp_df, emp_col, 'Employee Count')

    print(f"  Top 10 country summaries: {len(summaries_generated) - count_before}")

    # =====================================================================
    # MISSING CHARTS NOTEBOOK (from Missing_Charts_Analysis.ipynb)
    # =====================================================================

    print("\n--- Missing Charts Analysis summaries ---")

    # FA year-by-year (4 charts)
    print("\nGenerating Missing Charts FA summaries...")
    count_before = len(summaries_generated)

    generate_summary('global_fa_yearly',
        summarize_funding_yearly_bar, analysis_df, 'amount', None, 'Global', FUNDED_START_YEAR)
    generate_summary('americas_fa_yearly',
        summarize_funding_yearly_bar, analysis_df, 'amount', scope_americas, 'Americas', FUNDED_START_YEAR)
    generate_summary('asia_pacific_fa_yearly',
        summarize_funding_yearly_bar, analysis_df, 'amount', scope_asia_pacific, 'Asia/Pacific', FUNDED_START_YEAR)
    generate_summary('europe_africa_fa_yearly',
        summarize_funding_yearly_bar, analysis_df, 'amount', scope_europe_africa, 'Europe/Africa', FUNDED_START_YEAR)

    print(f"  Missing FA summaries: {len(summaries_generated) - count_before}")

    # FE year-by-year (4 charts)
    print("\nGenerating Missing Charts FE summaries...")
    count_before = len(summaries_generated)

    generate_summary('global_fe_yearly',
        summarize_funding_yearly_bar, all_events_df, 'count', None, 'Global', FUNDED_START_YEAR)
    generate_summary('americas_fe_yearly',
        summarize_funding_yearly_bar, all_events_df, 'count', scope_americas, 'Americas', FUNDED_START_YEAR)
    generate_summary('asia_pacific_fe_yearly',
        summarize_funding_yearly_bar, all_events_df, 'count', scope_asia_pacific, 'Asia/Pacific', FUNDED_START_YEAR)
    generate_summary('europe_africa_fe_yearly',
        summarize_funding_yearly_bar, all_events_df, 'count', scope_europe_africa, 'Europe/Africa', FUNDED_START_YEAR)

    print(f"  Missing FE summaries: {len(summaries_generated) - count_before}")

    # Companies Founded by Year (bucketed, 3 charts)
    print("\nGenerating Missing Charts companies-founded summaries...")
    count_before = len(summaries_generated)

    generate_summary('global_companies_founded_by_year',
        summarize_companies_founded_buckets, companies_df, None, 'Global')
    generate_summary('americas_companies_founded_by_year',
        summarize_companies_founded_buckets, companies_df, scope_americas, 'Americas')
    generate_summary('usa_companies_founded_by_year',
        summarize_companies_founded_buckets, companies_df, scope_usa, 'USA')

    print(f"  Missing companies-founded summaries: {len(summaries_generated) - count_before}")

    # Stacked Funded by Year (2 charts)
    print("\nGenerating Missing Charts stacked-funded summaries...")
    count_before = len(summaries_generated)

    generate_summary('asia_pacific_companies_by_year_funded',
        summarize_companies_by_year_funded_stacked, companies_df, scope_asia_pacific, 'Asia/Pacific')
    generate_summary('europe_africa_companies_by_year_funded',
        summarize_companies_by_year_funded_stacked, companies_df, scope_europe_africa, 'Europe/Africa')

    print(f"  Missing stacked-funded summaries: {len(summaries_generated) - count_before}")

    # Regional Status Bars (2 charts)
    print("\nGenerating Missing Charts regional status summaries...")
    count_before = len(summaries_generated)

    generate_summary('regional_deadpooled_percentage',
        summarize_regional_status_bars, companies_df, 'deadpooled', 'Global')
    generate_summary('regional_funded_percentage',
        summarize_regional_status_bars, companies_df, 'funded', 'Global')

    print(f"  Missing regional status summaries: {len(summaries_generated) - count_before}")

    # Funded Region Pie (1 chart)
    print("\nGenerating Missing Charts funded pie summary...")
    count_before = len(summaries_generated)

    generate_summary('regional_funded_companies_pie',
        summarize_funded_region_pie, companies_df, 'Global')

    print(f"  Missing funded pie summaries: {len(summaries_generated) - count_before}")

    # Top 10 from Missing Charts (Seed+A and Employee Count)
    # Already covered in the main Top 10 section above

# --- Global VC Chart (optional, requires separate CSV upload) ---
if not SKIP_VC_UPLOAD:
    print("\n--- Global VC Chart Summary ---")
    print("Upload the global VC totals CSV (e.g., global_vc_totals_2000_to_2025.csv).")
    vc_uploaded = files.upload()
    vc_filenames = list(vc_uploaded.keys())
    assert len(vc_filenames) == 1, f"Expected 1 CSV file, got {len(vc_filenames)}: {vc_filenames}"

    vc_df = pd.read_csv(vc_filenames[0])
    assert 'Year' in vc_df.columns, f"CSV missing 'Year' column. Columns: {list(vc_df.columns)}"
    VC_AMOUNT_COL = 'Global VC Investment (USD billions)'
    assert VC_AMOUNT_COL in vc_df.columns, f"CSV missing '{VC_AMOUNT_COL}'. Columns: {list(vc_df.columns)}"

    generate_summary('global_vc_totals',
        summarize_global_vc, vc_df, 2000, 'Global')

# --- Write Single Output File ---
os.makedirs(CHART_OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(CHART_OUTPUT_DIR, 'chart_summaries.txt')

with open(output_path, 'w') as out:
    # --- Highlights section at top ---
    all_highlights = []
    for chart_name, lines, highlights in all_summaries:
        for hl in highlights:
            all_highlights.append((chart_name, hl))

    if all_highlights:
        out.write('=' * 70 + '\n')
        out.write('HIGHLIGHTS — Striking Findings\n')
        out.write('=' * 70 + '\n\n')
        current_chart = None
        for chart_name, hl in all_highlights:
            if chart_name != current_chart:
                out.write(f'[{chart_name}]\n')
                current_chart = chart_name
            out.write(hl + '\n')
        out.write('\n')

    # --- Full summaries ---
    out.write('=' * 70 + '\n')
    out.write('FULL CHART SUMMARIES\n')
    out.write('=' * 70 + '\n\n')

    for chart_name, lines, highlights in all_summaries:
        # Mark highlighted lines inline with ** prefix
        hl_set = set(highlights)
        out.write('-' * 50 + '\n')
        for line in lines:
            starred = line.replace('- ', '** ', 1)
            if starred in hl_set:
                out.write(starred + '\n')
            else:
                out.write(line + '\n')
        out.write('\n')

print(f'Wrote {output_path}')
print(f'  {len(all_summaries)} chart summaries')
print(f'  {len(all_highlights)} highlights')

# --- Console Summary ---
print(f"\n{'='*60}")
print(f'Chart Summary Generation Complete')
print(f"{'='*60}")
print(f'Generated: {len(summaries_generated)} summaries')
print(f'Failed: {len(summaries_failed)} summaries')
if summaries_failed:
    print(f'\nFailed summaries:')
    for name, error in summaries_failed:
        print(f'  - {name}: {error}')
print(f"\nOutput: {output_path}")
print(f"{'='*60}")
